# Parsing GeoGebra Construction Protocolto Polars DataFrame

In [ ]:
# this magic for develop only
%load_ext autoreload
%autoreload 2

In [1]:
from ggblab import GeoGebra

In [2]:
# initialize base class, not open GeoGebra Widget
ggb = GeoGebra()

Using local cached file: xsd/common.xsd


In [3]:
from ggblab_extra.construction_io import ConstructionIO

In [4]:
await ggb.init()

In [5]:
c = ggb.file.load('2025_13_01.ggb')

In [6]:
r = await ggb.function("setBase64", [ggb.construction.base64_buffer.decode('utf-8')])

In [7]:
# df1 from a file df2 from a applet
df1 = await ConstructionIO.initialize_dataframe(ggb, file='2025_13_01.ggb')
df2 = await ConstructionIO.initialize_dataframe(ggb, use_applet=True)

In [8]:
df1

Name,Type,Command,Value,Caption,Layer,ShowObject,ShowLabel,Auxiliary
str,str,str,str,str,i64,bool,bool,bool
"""C""","""point""",null,null,null,9,true,true,false
"""A""","""point""",null,null,null,9,true,true,false
"""poly1""","""polygon""","""Polygon(C, A, 4)""",null,null,3,false,false,true
"""f""","""segment""","""Segment(C, A, poly1)""",null,null,2,false,false,true
"""g""","""segment""","""Segment(A, E, poly1)""",null,null,2,false,false,true
…,…,…,…,…,…,…,…,…
"""b_3""","""segment""","""Segment(P, B_1, t8)""",null,null,2,false,false,true
"""p_6""","""segment""","""Segment(B_1, A_1, t8)""",null,null,2,false,false,true
"""e_3""","""segment""","""Segment(A_1, P, t8)""",null,null,2,false,false,true


In [9]:
df2

Name,Type,Command,Value,Caption,Layer,ShowObject,ShowLabel,Auxiliary
str,str,str,str,str,i64,bool,bool,bool
"""C""","""point""",null,"""C = (0, 0)""",null,9,true,true,false
"""A""","""point""",null,"""A = (2.2, 0)""",null,9,true,true,false
"""poly1""","""polygon""","""Polygon(C, A, 4)""","""poly1 = 4.6""",null,3,false,false,true
"""f""","""segment""","""Segment(C, A, poly1)""","""f = 2.2""",null,2,false,false,true
"""g""","""segment""","""Segment(A, E, poly1)""","""g = 2.2""",null,2,false,false,true
…,…,…,…,…,…,…,…,…
"""b_3""","""segment""","""Segment(P, B_1, t8)""","""b_3 = 3""",null,2,false,false,true
"""p_6""","""segment""","""Segment(B_1, A_1, t8)""","""p_6 = 2.4""",null,2,false,false,true
"""e_3""","""segment""","""Segment(A_1, P, t8)""","""e_3 = 1.8""",null,2,false,false,true


In [10]:
set(df1["Type"].unique()) - set(df2["Type"].unique())

{'conic'}

In [11]:
set(df2["Type"].unique()) - set(df1["Type"].unique())

{'circle', 'quadrilateral', 'triangle'}

In [12]:
# df1 and df2 have different order...
# df1["Command"] == df2["Command"]
mask = df1['Command'].eq_missing(df2['Command']).not_()
# df1.filter(mask)
df2.filter(mask)

Name,Type,Command,Value,Caption,Layer,ShowObject,ShowLabel,Auxiliary
str,str,str,str,str,i64,bool,bool,bool
"""E""","""point""","""Polygon(C, A, 4)""","""E = (2.1, 2.2)""",null,2,true,false,true
"""D""","""point""","""Polygon(C, A, 4)""","""D = (0, 2.2)""",null,2,true,false,true
"""h""","""segment""","""Segment(E, D, poly1)""","""h = 2.2""",null,2,false,false,true
"""i""","""segment""","""Segment(D, C, poly1)""","""i = 2.2""",null,2,false,false,true
"""G""","""point""","""Polygon(A, C, 4)""","""G = (0, -2.2)""",null,4,true,false,true
…,…,…,…,…,…,…,…,…
"""H_1""","""point""","""Polygon(O', P, 4)""","""H_1 = (-0.3, 1)""",null,2,false,true,true
"""I_1""","""point""","""Polygon(O', P, 4)""","""I_1 = (-0.3, 0)""",null,2,false,true,true
"""f_4""","""segment""","""Segment(H_1, I_1, poly8)""","""f_4 = 1""",null,2,false,false,true


## IR files

In [ ]:
import os
os.path.splitext(ggb.file.source_file)[0]+'.json'

In [ ]:
df1.write_json(os.path.splitext(ggb.file.source_file)[0]+'.json')

In [ ]:
import xml.etree.ElementTree as ET
import xmltodict

In [ ]:
root = ET.Element(c.geogebra_xml)
tree = ET.ElementTree(root)
tree.write(os.path.splitext(ggb.file.source_file)[0]+'.xml', encoding='utf-8', xml_declaration=True)

## handle no root returens from the applet

In [13]:
r = await ggb.function("getXML", ["text1"])

In [14]:
print(r)

<expression label="text1" exp="&quot;1.  Thales&apos;s  theorem: right  triangle  inscribed  in  a  circle&quot;"/>
<element type="text" label="text1">
	<show object="true" label="false" ev="40"/>
	<objColor r="0" g="0" b="0" alpha="0"/>
	<layer val="9"/>
	<labelMode val="0"/>
	<isLaTeX val="true"/>
	<font serif="false" sizeM="1" size="0" style="0"/>
	<absoluteScreenLocation x="50" y="50"/>
</element>



In [15]:
import xml.etree.ElementTree as ET
from itertools import chain

try:
    o3 = ggb.file.ggb_schema.decode(r)
except ET.ParseError:
    vr = ET.fromstringlist(chain(['<construction>'], r, ['</construction>']))
    o3 = ggb.file.ggb_schema.decode(ET.tostring(vr).decode('utf-8'))
o3

{'expression': [{'@label': 'text1',
   '@exp': '"1. \xa0Thales\'s \xa0theorem: right \xa0triangle \xa0inscribed \xa0in \xa0a \xa0circle"'}],
 'element': [{'@type': 'text',
   '@label': 'text1',
   'show': [{'@object': True, '@label': False, '@ev': 40}],
   'objColor': [{'@r': 0, '@g': 0, '@b': 0, '@alpha': 0.0}],
   'layer': [{'@val': 9}],
   'labelMode': [{'@val': 0}],
   'isLaTeX': [{'@val': True}],
   'font': [{'@serif': False, '@sizeM': 1.0, '@size': 0, '@style': 0}],
   'absoluteScreenLocation': [{'@x': 50.0, '@y': 50.0}]}]}

In [16]:
o3['element'][0].get('show')

[{'@object': True, '@label': False, '@ev': 40}]